In [1]:
import langchain
import langchain_anthropic
import chromadb
from sentence_transformers import SentenceTransformer
import requests
import xml.etree.ElementTree as ET
print("Imports OK")

c:\Users\diego\anaconda3\envs\rag-fds\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
#We extract the 5 most recent papers in the cs.
# We extract the title and abstract of each paper. 

url = "http://export.arxiv.org/api/query?search_query=cat:cs.CL&start=0&max_results=5"
response = requests.get(url)
root = ET.fromstring(response.content)
ns = {"atom": "http://www.w3.org/2005/Atom"}

papers = []
for entry in root.findall("atom:entry", ns):
    title = entry.find("atom:title", ns).text.strip()
    summary = entry.find("atom:summary", ns).text.strip()
    papers.append({"title": title, "abstract": summary})

for p in papers:
    print("-", p["title"][:80])

- End-to-End Speaker Diarization as Post-Processing
- Regularized Attentive Capsule Network for Overlapped Relation Extraction
- Should I visit this place? Inclusion and Exclusion Phrase Mining from Reviews
- Speech Synthesis as Augmentation for Low-Resource ASR
- QUACKIE: A NLP Classification Task With Ground Truth Explanations


In [ ]:
# We use the BGE model to encode the abstracts of the papers. 
# The expected output is a 2D array with shape (5, 384), where 5 is the number of papers and 384 is the dimensionality of the embeddings.
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
texts = [p["abstract"] for p in papers]
embeddings = model.encode(texts)
print(embeddings.shape) 

c:\Users\diego\anaconda3\envs\rag-fds\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diego\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2036.15it/s]


(5, 384)


In [ ]:
# We create a ChromaDB collection and add the papers to it.

client = chromadb.Client() 
collection = client.create_collection("smoke_test")

collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=[f"paper_{i}" for i in range(len(texts))],
)

query_vec = model.encode(["transformer models for language understanding"])
results = collection.query(query_embeddings=query_vec.tolist(), n_results=2)
print(results["documents"])

[['NLP Interpretability aims to increase trust in model predictions. This makes evaluating interpretability approaches a pressing issue. There are multiple datasets for evaluating NLP Interpretability, but their dependence on human provided ground truths raises questions about their unbiasedness. In this work, we take a different approach and formulate a specific classification task by diverting question-answering datasets. For this custom classification task, the interpretability ground-truth arises directly from the definition of the classification problem. We use this method to propose a benchmark and lay the groundwork for future research in NLP interpretability by evaluating a wide range of current state of the art methods.', 'Speech synthesis might hold the key to low-resource speech recognition. Data augmentation techniques have become an essential part of modern speech recognition training. Yet, they are simple, naive, and rarely reflect real-world conditions. Meanwhile, speech

In [6]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", max_tokens=50)
response = llm.invoke("Responde solo con 'pipeline funcionando' si recibes este mensaje.")
print(response.content)

pipeline funcionando


In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

text = (
    "Self-attention mechanisms enable transformers to model "
    "long-range dependencies efficiently across sequential data "
    "without relying on recurrence or convolutional operations."
)

chars = len(text)
tokens = len(tokenizer.encode(text))

print(f"Caracteres (len()):        {chars}")
print(f"Tokens (tokenizer real):   {tokens}")
print(f"Ratio: {chars / tokens:.1f} caracteres por token")

c:\Users\diego\anaconda3\envs\rag-fds\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Caracteres (len()):        172
Tokens (tokenizer real):   33
Ratio: 5.2 caracteres por token


In [1]:
import csv

with open("eval/ragas_results.csv") as f:
    rows = list(csv.DictReader(f))

# Las 3 peores en faithfulness
worst_faith = sorted(rows, key=lambda r: float(r["faithfulness"]))[:3]
print("--- Peores en faithfulness ---")
for r in worst_faith:
    print(f"{r['faithfulness']} | {r['question'][:80]}")

# Las 3 peores en answer_relevancy
worst_rel = sorted(rows, key=lambda r: float(r["answer_relevancy"]))[:3]
print("\n--- Peores en answer_relevancy ---")
for r in worst_rel:
    print(f"{r['answer_relevancy']} | {r['question'][:80]}")

--- Peores en faithfulness ---
0.6666666666666666 | According to the sociolinguistic analyses in the study, which group of COVID-19 
0.6666666666666666 | What accuracy did the best model achieve on Popoluca and Tepehua, respectively, 
0.6666666666666666 | What BLEU score does NABU achieve on the English monolingual RDF-to-text task, o

--- Peores en answer_relevancy ---
0.0 | By how much did the self-supervised model outperform the best performing models 
0.0 | What accuracy did the best model achieve on Popoluca and Tepehua, respectively, 
0.0 | What efficiency improvement threshold does the proposed topic modeling-based app
